# Quickstart: the forecast-price feedback loop

Run the greedy baseline and the control-function policy on the same
single-product world and compare how well each recovers the true elasticity.
This uses the library API directly (no dashboard).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fploop.forecasters.gbt import GBTForecaster
from fploop.generators.linear_logit import LinearLogitWorld
from fploop.loop import run_simulation
from fploop.policies import ControlFunctionPolicy, GreedyBaseline
from fploop.types import WorldConfig

In [ ]:
config = WorldConfig(
    elasticity=-1.5,
    endogeneity_strength=0.6,
    shock_std=0.2,
    cost_shifter_std=0.15,
    horizon=200,
)
seed = 0


def run(policy_cls):
    world = LinearLogitWorld(config)
    policy = policy_cls(GBTForecaster(), rng=np.random.default_rng(seed))
    return run_simulation(world, policy, seed=seed)


greedy = run(GreedyBaseline)
causal = run(ControlFunctionPolicy)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(config.elasticity, color="black", lw=2, label="true elasticity")
ax.plot(greedy.estimated_elasticity.ravel(), color="gray", label="greedy (Arm 0)")
ax.plot(causal.estimated_elasticity.ravel(), color="purple", label="control function")
ax.set_xlabel("pricing cycle")
ax.set_ylabel("estimated elasticity")
ax.legend()
ax.set_title("Greedy spirals away from beta; the control function tracks it")
plt.show()